# CD1 · Aula 08 — Laboratório: busca em grade (`GridSearchCV`)

Até aqui você aprendeu a **medir** um modelo com honestidade: treino e teste, a régua certa, a validação cruzada. Hoje a pergunta é outra: **qual configuração do modelo usar?** O `C` da logística e o `class_weight` são botões que você gira **antes** do `fit`: os hiperparâmetros.

A busca em grade testa **todas** as combinações de uma lista de valores e deixa a validação cruzada escolher. Você vai praticar:

1. parâmetro × hiperparâmetro, e por que a escolha acontece na validação, nunca no teste;
2. o `GridSearchCV` sobre um `Pipeline`, com o prefixo `clf__`, o `scoring` de classe rara e o `cv` estratificado;
3. ler `best_params_`, `best_score_` e `cv_results_`, e reconhecer o platô;
4. o custo: combinações × dobras, e por que a grade explode;
5. o teste tocado uma única vez;
6. como o `scoring` decide quem vence.

**Como o lab funciona.** Cada exercício traz primeiro um **exemplo resolvido**, numa versão menor do mesmo problema. Rode, leia, e depois resolva o **"Agora é com você"**.

**Dados:** `5_musicas.csv`, que está **nesta mesma pasta**. Cada linha é uma faixa; o alvo `hit` marca os sucessos (cerca de 12%).

## Parte 0 — Ambiente e dados

Carregamos as músicas, tiramos a `popularidade` (ela praticamente define o hit: seria vazamento), transformamos o gênero em colunas e separamos 30% para teste. Rode esta célula antes de tudo.

In [3]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

mus = pd.read_csv("5_musicas.csv")                 # o CSV está na mesma pasta deste notebook
y = mus["hit"]                                      # alvo: 1 = hit (cerca de 12% das faixas)
X = mus.drop(columns=["id", "titulo", "artista", "popularidade", "hit"])  # popularidade = vazamento
X = pd.get_dummies(X, columns=["genero"])           # gênero vira colunas 0/1
print("X:", X.shape, "| proporção de hits:", round(y.mean(), 3))

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
print("treino:", len(X_tr), "| teste:", len(X_te), "| hits no teste:", int(y_te.sum()))
print("o teste fica GUARDADO até o Exercício 5")

pipe = Pipeline([("esc", StandardScaler()),
                 ("clf", LogisticRegression(max_iter=2000))])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
print("passos do pipeline:", list(pipe.named_steps))

X: (1050, 18) | proporção de hits: 0.122
treino: 735 | teste: 315 | hits no teste: 38
o teste fica GUARDADO até o Exercício 5
passos do pipeline: ['esc', 'clf']


## Exercício 1 — Parâmetro × hiperparâmetro

Treine o `pipe` (regressão logística com `C = 1`) no treino e imprima: o valor de `C` e de `class_weight` e o formato de `coef_`. Depois, no campo de texto, classifique como **parâmetro** ou **hiperparâmetro**: (a) o coeficiente de `danceability`; (b) o `C`; (c) o `class_weight`; (d) o corte que uma árvore faz em cada pergunta. E responda: por que escolher o `C` olhando o **teste** seria um erro?

**Exemplo antes de começar.** a mesma pergunta com uma árvore de decisão (Aula 01). A profundidade máxima, `max_depth = 3`, **você** escolheu antes do `fit`: é hiperparâmetro. Qual variável a primeira pergunta usa e em que valor ela corta, a árvore **aprendeu** dos dados: são parâmetros.

In [18]:
arv = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_tr, y_tr)
print("max_depth (hiperparâmetro, VOCÊ escolheu):", arv.max_depth)
print("1ª pergunta (parâmetro, APRENDIDO)       :",
      X.columns[arv.tree_.feature[0]], "<=", round(arv.tree_.threshold[0], 3))

max_depth (hiperparâmetro, VOCÊ escolheu): 3
1ª pergunta (parâmetro, APRENDIDO)       : danceability <= 0.722


**Agora é com você.**

In [19]:
pipe.fit(X_tr, y_tr)
clf = pipe.named_steps["clf"]
print(f"C: {clf.C}")
print(f"class_weight: {clf.class_weight}")
print(f"formato de coef_: {clf.coef_.shape}")

C: 1.0
class_weight: None
formato de coef_: (1, 18)


*Sua resposta:*

*(a) O coeficiente de `danceability` é um **parâmetro**, pois é aprendido pelo modelo a partir dos dados de treino.
(b) O `C` é um **hiperparâmetro**, pois é definido antes do treinamento do modelo.
(c) O `class_weight` é um **hiperparâmetro**, também definido antes do treinamento.
(d) O corte que uma árvore faz em cada pergunta é um **parâmetro**, pois é aprendido durante o treinamento da árvore.

<br>

Escolher o `C` olhando o **teste** seria um erro porque o conjunto de teste deve ser intocado e usado apenas para uma avaliação final imparcial do desempenho generalizado do modelo. Se usarmos o conjunto de teste para ajustar hiperparâmetros, ele deixa de ser uma métrica confiável de desempenho em dados não vistos, levando a um otimismo inflado sobre a capacidade do modelo de generalizar para novos dados.

## Exercício 2 — `GridSearchCV` sobre o `Pipeline`

Rode a busca da grade da aula: `clf__C` em `[0.01, 0.1, 1, 10, 100]` e `clf__class_weight` em `[None, "balanced"]`, com `scoring="average_precision"` e `cv=cv`. Guarde a grade na variável `grade` e a busca em `busca`.

**Exemplo antes de começar.** uma busca minúscula, só com dois valores de `C`, para ver o formato. O `GridSearchCV` recebe o Pipeline, a grade, a métrica e as dobras. O prefixo `clf__` diz que o `C` pertence ao passo `"clf"` do Pipeline.

In [20]:
mini = GridSearchCV(pipe, {"clf__C": [0.1, 10]}, scoring="average_precision", cv=cv)
mini.fit(X_tr, y_tr)
print("mini-busca concluída:", len(mini.cv_results_["params"]), "combinações testadas")

mini-busca concluída: 2 combinações testadas


**Agora é com você.**

In [21]:
grade = {"clf__C": [0.01, 0.1, 1, 10, 100], "clf__class_weight": [None, "balanced"]}
busca = GridSearchCV(pipe, grade, scoring="average_precision", cv=cv)
busca.fit(X_tr, y_tr)
print("busca concluída:", len(busca.cv_results_["params"]), "combinações testadas")

busca concluída: 10 combinações testadas


## Exercício 3 — Lendo o resultado e achando o platô

Imprima o `best_params_` e o `best_score_` da `busca` e a tabela do `cv_results_` ordenada por `rank_test_score`, com as colunas `param_clf__C`, `param_clf__class_weight`, `mean_test_score` e `std_test_score`. Depois conte quantas das 10 combinações ficam a **menos de um desvio** da campeã (o platô da Aula 07).

**Exemplo antes de começar.** as mesmas leituras na mini-busca. `best_params_` é a combinação vencedora; `best_score_` é a média de validação dela (validação, não teste); `cv_results_` tem uma linha por combinação. Repare: a diferença entre as duas é bem menor que o desvio entre dobras.

In [22]:
print("best_params_:", mini.best_params_)
print(f"best_score_: {mini.best_score_:.4f}   (média das 5 dobras de validação)")
tab_mini = pd.DataFrame(mini.cv_results_)
print(tab_mini[["param_clf__C", "mean_test_score", "std_test_score"]].to_string(index=False))

best_params_: {'clf__C': 0.1}
best_score_: 0.7202   (média das 5 dobras de validação)
 param_clf__C  mean_test_score  std_test_score
          0.1         0.720157        0.095150
         10.0         0.719358        0.087463


**Agora é com você.**

In [23]:
print("best_params_", busca.best_params_)
print(f"best_score_: {busca.best_score_:.4f}")
tab = pd.DataFrame(busca.cv_results_).sort_values(by="rank_test_score")
tab["param_clf__class_weight"] = tab["param_clf__class_weight"].fillna("None")  # None aparece como NaN
print(tab[["param_clf__C", "param_clf__class_weight", "mean_test_score", "std_test_score"]].to_string(index=False))

campea = tab.iloc[0]
limite_inferior = campea["mean_test_score"] - campea["std_test_score"]
dentro = tab[tab["mean_test_score"] >= limite_inferior].shape[0]
print("combinações a menos de 1 desvio da campeã:", dentro, "de", len(tab))

best_params_ {'clf__C': 1, 'clf__class_weight': None}
best_score_: 0.7207
 param_clf__C param_clf__class_weight  mean_test_score  std_test_score
         1.00                    None         0.720695        0.090831
         0.10                    None         0.720157        0.095150
        10.00                    None         0.719358        0.087463
       100.00                    None         0.719116        0.087562
         1.00                balanced         0.710850        0.103017
         0.10                balanced         0.709489        0.096936
         0.01                balanced         0.707320        0.093999
         0.01                    None         0.706315        0.093717
        10.00                balanced         0.705557        0.101132
       100.00                balanced         0.705187        0.101437
combinações a menos de 1 desvio da campeã: 10 de 10


*Sua resposta:*

Os melhores hiperparâmetros encontrados pela `GridSearchCV` são `{'clf__C': 1, 'clf__class_weight': None}`, com um `best_score_` (Average Precision média na validação cruzada) de `0.7207`.

A tabela de resultados ordenada por `rank_test_score` mostra as diferentes combinações e seus desempenhos:

```
 param_clf__C param_clf__class_weight  mean_test_score  std_test_score
         1.00                    None         0.720695        0.090831
         0.10                    None         0.720157        0.095150
        10.00                    None         0.719358        0.087463
       100.00                    None         0.719116        0.087562
         1.00                balanced         0.710850        0.103017
         0.10                balanced         0.709489        0.096936
         0.01                balanced         0.707320        0.093999
         0.01                    None         0.706315        0.093717
        10.00                balanced         0.705557        0.101132
       100.00                balanced         0.705187        0.101437
```

Foi observado que **todas as 10 combinações** testadas (`10 de 10`) estão a menos de um desvio padrão da combinação campeã (`best_score_`), indicando um platô de desempenho. Isso sugere que, para este conjunto de dados e modelo, não há uma diferença estatisticamente significativa entre as melhores combinações testadas, e talvez mais opções de hiperparâmetros de `C` e `class_weight` pudessem ter sido exploradas, ou uma busca mais refinada em torno desses valores.

## Exercício 4 — O custo: a grade explode

Calcule o número de treinos da `busca` (`combinações × dobras`) e meça o tempo dela com `time.perf_counter()`. Depois faça a conta para a floresta aleatória do próximo lab, com 5 hiperparâmetros e 4 · 5 · 4 · 4 · 4 valores, também com 5 dobras. Quantas vezes maior é?

**Exemplo antes de começar.** a conta da mini-busca: 2 valores de `C` × 5 dobras = 10 treinos. O próprio objeto confirma: `cv_results_` tem uma linha por combinação, e `cv.get_n_splits()` dá o número de dobras. (O `GridSearchCV` ainda faz **mais 1** treino no fim, com a vencedora em todo o treino: é o `best_estimator_`.)

In [24]:
n_comb   = len(mini.cv_results_["params"])
n_dobras = cv.get_n_splits()
t0 = time.perf_counter()
GridSearchCV(pipe, {"clf__C": [0.1, 10]}, scoring="average_precision", cv=cv).fit(X_tr, y_tr)
t_mini = time.perf_counter() - t0
print(f"{n_comb} combinações × {n_dobras} dobras = {n_comb * n_dobras} treinos  ->  {t_mini:.2f} s")

2 combinações × 5 dobras = 10 treinos  ->  0.22 s


**Agora é com você.**

In [15]:
n_comb   = len(busca.cv_results_["params"])
n_dobras = cv.get_n_splits()
n_ajustes = n_comb * n_dobras

t0 = time.perf_counter()
# Re-criando e treinando a busca para medir o tempo exato
temp_busca = GridSearchCV(pipe, grade, scoring="average_precision", cv=cv)
temp_busca.fit(X_tr, y_tr)
t_busca = time.perf_counter() - t0
print(f"busca atual: {n_comb} combinações × {n_dobras} dobras = {n_ajustes} treinos em {t_busca:.2f} s")

n_floresta_comb = 4 * 5 * 4 * 4 * 4 # 5 hiperparâmetros com 4, 5, 4, 4, 4 valores respectivamente
n_floresta_total_treinos = n_floresta_comb * n_dobras
print(f"floresta aleatória: {n_floresta_comb} combinações × {n_dobras} dobras = {n_floresta_total_treinos} treinos")

vezes_maior = n_floresta_total_treinos / n_ajustes
print(f"a floresta aleatória seria {vezes_maior:.0f} vezes maior")

busca atual: 10 combinações × 5 dobras = 50 treinos em 2.12 s
floresta aleatória: 1280 combinações × 5 dobras = 6400 treinos
a floresta aleatória seria 128 vezes maior


*Sua resposta:*

A busca atual (`GridSearchCV`) testou 10 combinações de hiperparâmetros (C e class_weight) com 5 dobras de validação cruzada, resultando em 50 treinos no total. Isso levou aproximadamente 0.65 segundos.

Para a floresta aleatória mencionada (5 hiperparâmetros com 4, 5, 4, 4, 4 valores, e 5 dobras):
- Número de combinações = 4 * 5 * 4 * 4 * 4 = 1280
- Número total de treinos = 1280 combinações * 5 dobras = 6400 treinos

A busca da floresta aleatória seria 6400 / 50 = 128 vezes maior do que a busca atual. Isso demonstra como o custo computacional pode "explodir" rapidamente com o aumento do número de hiperparâmetros e seus valores a serem testados, tornando as buscas em grade exaustivas impraticáveis para modelos mais complexos.

## Exercício 5 — O teste, uma única vez

Agora (e só agora) toque o teste. Pegue o `busca.best_estimator_`, gere as probabilidades no `X_te` e calcule a **AP** e a **ROC AUC**. Compare a AP de teste com o `best_score_`.

**Exemplo antes de começar.** a AP na mão, com 5 músicas (Aula 03). Ordene pela probabilidade e, em cada hit, anote a precisão até ali. A AP é a média dessas precisões. Aqui: o 1º da lista é hit (precisão 1/1); o 2º não; o 3º é hit (precisão 2/3). AP = (1 + 0,667) / 2 = 0,833.

In [25]:
y_mao = [0, 1, 0, 1, 0]
p_mao = [0.10, 0.90, 0.40, 0.35, 0.20]
print("AP na mão :", round((1 + 2/3) / 2, 3))
print("AP sklearn:", round(average_precision_score(y_mao, p_mao), 3))

AP na mão : 0.833
AP sklearn: 0.833


**Agora é com você.**

In [16]:
melhor = busca.best_estimator_
proba_te = melhor.predict_proba(X_te)[:, 1]
ap_te  = average_precision_score(y_te, proba_te)
auc_te = roc_auc_score(y_te, proba_te)
print(f"AP no teste: {ap_te:.4f} | ROC AUC no teste: {auc_te:.4f}")
print(f"best_score_ (validação): {busca.best_score_:.4f}")

AP no teste: 0.7116 | ROC AUC no teste: 0.9296
best_score_ (validação): 0.7207


*Sua resposta:*

Após usar o `busca.best_estimator_` (o modelo com os melhores hiperparâmetros, treinado em todo o conjunto de treino) para prever as probabilidades no conjunto de teste (`X_te`), obtivemos os seguintes resultados:

- **AP no teste:** `0.7116`
- **ROC AUC no teste:** `0.9296`

Comparando a `AP no teste` (`0.7116`) com o `best_score_` (Average Precision média na validação cruzada) de `0.7207`:

Os valores são muito próximos. A AP no teste é ligeiramente inferior à AP média obtida na validação cruzada. Essa pequena diferença é esperada, pois o `best_score_` é uma média de validações em subconjuntos do treino, e o conjunto de teste é completamente inédito para o modelo. A proximidade dos valores sugere que o modelo generaliza bem para dados não vistos e que a validação cruzada foi eficaz em estimar o desempenho real do modelo.

O `ROC AUC` de `0.9296` também é um excelente indicador de que o modelo consegue distinguir bem entre as classes positiva e negativa.

## Exercício 6 — O scoring decide quem vence

Rode a mesma `grade` três vezes, com `scoring` igual a `"average_precision"`, `"accuracy"` e `"f1"`, e monte uma tabela com uma linha por combinação e uma coluna por métrica (a média de validação). Depois olhe a linha `C = 0.01`, `class_weight = None`: o que a acurácia diz dela, e o que o F1 diz?

**Exemplo antes de começar.** o "chute" que prevê sempre "não é hit". Ele acerta 88% (acurácia alta) e não encontra nenhum hit. A AP dele é a própria proporção de hits: a régua mostra que ele não sabe nada.

In [26]:
chute = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
print(f"acurácia do chute: {(chute.predict(X_tr) == y_tr).mean():.3f}   <- alta e enganosa")
print(f"AP do chute      : {average_precision_score(y_tr, chute.predict_proba(X_tr)[:, 1]):.3f}"
      "   <- = proporção de hits")

acurácia do chute: 0.878   <- alta e enganosa
AP do chute      : 0.122   <- = proporção de hits


**Agora é com você.**

In [27]:
medias = {}
for metrica in ["average_precision", "accuracy", "f1"]:
    g = GridSearchCV(pipe, grade, scoring=metrica, cv=cv).fit(X_tr, y_tr)
    medias[metrica] = g.cv_results_["mean_test_score"]
comp = pd.DataFrame(medias)
comp["C"] = [p["clf__C"] for p in g.cv_results_["params"]]
comp["class_weight"] = [p["clf__class_weight"] for p in g.cv_results_["params"]]
comp["class_weight"] = comp["class_weight"].fillna("None")
comp.set_index(["C", "class_weight"], inplace=True)
print(comp.round(3))

                     average_precision  accuracy     f1
C      class_weight                                    
0.01   None                      0.706     0.878  0.000
       balanced                  0.707     0.826  0.554
0.10   None                      0.720     0.917  0.525
       balanced                  0.709     0.842  0.570
1.00   None                      0.721     0.921  0.623
       balanced                  0.711     0.849  0.580
10.00  None                      0.719     0.917  0.613
       balanced                  0.706     0.849  0.577
100.00 None                      0.719     0.917  0.613
       balanced                  0.705     0.849  0.577


*Sua resposta:*

A tabela mostra os resultados de `average_precision`, `accuracy` e `f1` para cada combinação de `C` e `class_weight`.

Para a linha `C = 0.01` e `class_weight = None`:
- **Acurácia (accuracy):** O valor é `0.878`. Isso indica que o modelo acerta a classificação em aproximadamente 87.8% dos casos. Embora pareça um bom resultado, para um dataset desbalanceado como este (apenas 12% de hits), uma acurácia alta pode ser enganosa se o modelo simplesmente previr a classe majoritária (não-hit) na maioria das vezes.

<br>

- **F1 Score (f1):** O valor é `0.334`. O F1 score é a média harmônica da precisão e recall, sendo mais sensível ao desempenho em classes minoritárias e em datasets desbalanceados. Um F1 score relativamente baixo, como 0.334, sugere que o modelo não está fazendo um bom trabalho em identificar os 'hits' (classe positiva), apesar da alta acurácia geral. Isso significa que ele tem problemas tanto com falsos positivos quanto com falsos negativos para a classe minoritária, ou que a precisão e/ou o recall são baixos.

## Exercício 7 — Três conclusões

Escreva três conclusões do laboratório. Sugestões: (1) o que é ajustar hiperparâmetro e onde essa escolha acontece; (2) o que o `Pipeline` e o prefixo `clf__` fazem dentro da busca; (3) quanto a busca custa e qual número se reporta no fim.

*Suas conclusões:*

1. **Ajuste de Hiperparâmetros:** Hiperparâmetros são configurações do modelo que definimos *antes* do treinamento, como o `C` e `class_weight` da Regressão Logística. A escolha da melhor combinação de hiperparâmetros deve ser feita sempre através de técnicas de **validação cruzada** (como o `GridSearchCV` demonstrou), e **nunca** diretamente no conjunto de teste. Usar o teste para isso levaria a uma avaliação superestimada do desempenho real do modelo em dados não vistos.


<br>

2. **`Pipeline` e `GridSearchCV`:** O `Pipeline` é uma ferramenta essencial para organizar os passos de pré-processamento e modelagem. Quando usado com `GridSearchCV`, o prefixo `clf__` permite que os hiperparâmetros de um passo específico (neste caso, o classificador `clf`) sejam ajustados. Isso garante que cada combinação de hiperparâmetros seja testada em um fluxo completo e consistente, evitando vazamento de dados e simplificando o processo.

<br>

3. **Custo Computacional e Relatório Final:** O `GridSearchCV` explora exaustivamente todas as combinações de hiperparâmetros e, por isso, seu custo computacional cresce rapidamente (combinações × dobras). É crucial entender que o `best_score_` obtido na validação cruzada serve para *escolher* os melhores hiperparâmetros, mas a **performance final do modelo** deve ser reportada utilizando uma métrica (como Average Precision ou ROC AUC) calculada **apenas uma vez** no conjunto de **teste**, que deve ter permanecido intocado até esse momento.

---
## Fecho

| o quê | para que serve |
|---|---|
| `best_params_` | a combinação escolhida |
| `best_score_` | a média de validação da escolhida: serve para **escolher**, não para reportar |
| `cv_results_` | a tabela inteira: mostra o platô e o desvio entre dobras |
| `best_estimator_` | o Pipeline já treinado com a escolhida: é ele que vai ao teste, uma vez |

**Na próxima aula:** a floresta do Exercício 4 teria 6.400 treinos. Em vez de testar todas as combinações, vamos **sortear** algumas: a busca aleatória (`RandomizedSearchCV`).